# n_bits vs MAE
Cross-validated MAE over fingerprint size.


In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

MODEL_DIR = 'random-forest'
MODEL_NAME = 'Random Forest'

cwd = Path.cwd()
project_root = Path(".." ).resolve() if cwd.name == MODEL_DIR else cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from qspr_config import (
    DATA_PATH,
    ECFP_RADIUS,
    ECFP_N_BITS,
    N_BITS_GRID,
    N_ESTIMATORS,
    N_ESTIMATORS_GRID,
    RANDOM_SEED,
    CV_FOLDS,
    N_JOBS,
    FIG_DPI,
    FIGSIZE_SQUARE,
)
from qspr_common import (
    load_dataset,
    build_feature_matrix,
    apply_plot_style,
    resolve_output_dir,
)

apply_plot_style()
out_dir = resolve_output_dir(MODEL_DIR)


In [2]:
df_raw = load_dataset(DATA_PATH)


In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor

cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

mean_scores = []
std_scores = []
for n_bits in N_BITS_GRID:
    df_bits, X = build_feature_matrix(df_raw, radius=ECFP_RADIUS, n_bits=n_bits)
    y = df_bits["Solubility"].to_numpy()

    maes = []
    for train_idx, test_idx in cv.split(X):
        model = RandomForestRegressor(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_SEED,
            n_jobs=N_JOBS,
        )
        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[test_idx])
        maes.append(np.mean(np.abs(pred - y[test_idx])))

    mean_scores.append(np.mean(maes))
    std_scores.append(np.std(maes))

fig, ax = plt.subplots(figsize=FIGSIZE_SQUARE)
ax.errorbar(N_BITS_GRID, mean_scores, yerr=std_scores, marker="o", capsize=3)
ax.set_xlabel("n_bits")
ax.set_ylabel("MAE")
ax.set_title(f"{MODEL_NAME}: n_bits vs MAE")
ax.set_xticks(N_BITS_GRID)
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

out_path = out_dir / "n_bits_vs_mae.png"
fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()
out_path
